In [ ]:
!pip install cmu-multimodal-sdk

In [ ]:

import kagglehub
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel
from pathlib import Path
import logging
import pickle
from sklearn.metrics import mean_absolute_error, accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import matplotlib.pyplot as plt
import torch.nn.functional as F
import seaborn as sns
import time
from mmsdk import mmdatasdk as md
import sys
import os

# Download latest version
path = kagglehub.dataset_download("samarwarsi/cmu-mosei")

print("Path to dataset files:", path)

# Cấu hình tham số
RAW_DATA_DIR = path + "/CMU-MOSEI/"
PROCESSED_DATA_DIR = "/content/processed_data"
DATASET_NAME = "CMU_MOSEI"
TEXT_MAX_LENGTH = 128
AUDIO_FEATURE_SIZE = 74
TEXT_EMBEDDING_DIM = 768
VISUAL_FEATURE_SIZE = 47
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
NUM_EPOCHS = 10
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5

# Thiết lập logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler('/content/logfile.log')
    ]
)
logger = logging.getLogger(__name__)

# Thiết lập seed
torch.manual_seed(SEED)
np.random.seed(SEED)

# Định nghĩa Dataset
class MOSEIDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data["labels"])

    def __getitem__(self, idx):
        return {
            "acoustic": torch.tensor(self.data["acoustic"][idx], dtype=torch.float32),
            "visual": torch.tensor(self.data["visual"][idx], dtype=torch.float32),
            "language": torch.tensor(self.data["language"][idx], dtype=torch.float32),
            "label": torch.tensor(self.data["labels"][idx], dtype=torch.float32)
        }

# Định nghĩa Preprocessor
class MOSEIPreprocessor:
    def __init__(self, raw_data_dir, processed_data_dir):
        self.raw_data_dir = Path(raw_data_dir)
        self.processed_data_dir = Path(processed_data_dir)
        self.processed_data_dir.mkdir(exist_ok=True, parents=True)

        self.tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
        self.bert_model = BertModel.from_pretrained("bert-base-uncased")
        self.bert_model.eval()
        self.device = DEVICE
        print(f"Using device: {self.device}")

        self.data = {
           "train": {"language": [], "acoustic": [], "visual": [], "labels": []},
            "val": {"language": [], "acoustic": [], "visual": [], "labels": []},
            "test": {"language": [], "acoustic": [], "visual": [],  "labels": []}
        }

        self.metadata = {
            "text_dim": TEXT_EMBEDDING_DIM,
            "audio_dim": AUDIO_FEATURE_SIZE,
            "visual_dim": VISUAL_FEATURE_SIZE,
            "num_classes": 1,
            "train_samples": 0,
            "val_samples": 0,
            "test_samples": 0
        }

    def load_aligned_data(self):
        print("Loading aligned data...")
        try:
            dataset = {
                "language": str(self.raw_data_dir / "languages" / "CMU_MOSEI_TimestampedWords.csd"),
                "acoustic": str(self.raw_data_dir / "acoustics" / "CMU_MOSEI_COVAREP.csd"),
                "visual": str(self.raw_data_dir / "visuals" / "CMU_MOSEI_VisualOpenFace2.csd"),
                "labels": str(self.raw_data_dir / "labels" / "CMU_MOSEI_Labels.csd")
            }
            for key, path in dataset.items():
                if not os.path.exists(path):
                    print(f"Error: File {path} does not exist")
                    return None
            mosei_dataset = md.mmdataset(dataset)
            print(f"Loaded dataset with {len(mosei_dataset['labels'].keys())} segments")
            return mosei_dataset
        except Exception as e:
            print(f"Error loading aligned data: {e}")
            return None

    def extract_text_features(self, text):
        try:
            inputs = self.tokenizer(
                text,
                padding="max_length",
                truncation=True,
                max_length=TEXT_MAX_LENGTH,
                return_tensors="pt"
            )
            inputs = {key: val.to(self.device) for key, val in inputs.items()}
            with torch.no_grad():
                outputs = self.bert_model(**inputs)
                embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            return embeddings[0]
        except Exception as e:
            logger.error(f"Error extracting text features: {e}")
            return np.zeros(TEXT_EMBEDDING_DIM)

    def extract_audio_features(self, audio_features):
        if len(audio_features.shape) > 1 and audio_features.shape[0] > 0:
            selected_features = np.mean(audio_features, axis=0)
            if np.any(np.isnan(selected_features)):
                selected_features = np.nan_to_num(selected_features, nan=0.0)
            if len(selected_features) > AUDIO_FEATURE_SIZE:
                selected_features = selected_features[:AUDIO_FEATURE_SIZE]
            elif len(selected_features) < AUDIO_FEATURE_SIZE:
                selected_features = np.pad(selected_features, (0, AUDIO_FEATURE_SIZE - len(selected_features)))
            return selected_features
        return np.zeros(AUDIO_FEATURE_SIZE)

    def extract_visual_features(self, visual_features):
        if len(visual_features.shape) > 1 and visual_features.shape[0] > 0:
            selected_features = np.mean(visual_features, axis=0)
            if np.any(np.isnan(selected_features)):
                selected_features = np.nan_to_num(selected_features, nan=0.0)
            if len(selected_features) > VISUAL_FEATURE_SIZE:
                selected_features = selected_features[:VISUAL_FEATURE_SIZE]
            elif len(selected_features) < VISUAL_FEATURE_SIZE:
                selected_features = np.pad(selected_features, (0, VISUAL_FEATURE_SIZE - len(selected_features)))
            return selected_features
        return np.zeros(VISUAL_FEATURE_SIZE)

    def process_dataset(self):
        print("Preprocessing dataset...")
        dataset = self.load_aligned_data()
        if dataset is None:
            print("Cannot load dataset")
            return False

        segment_ids = list(dataset["labels"].keys())
        print(f"Found {len(segment_ids)} segments in dataset")

        train_ids, test_ids = train_test_split(segment_ids, test_size=0.2, random_state=SEED)
        train_ids, val_ids = train_test_split(train_ids, test_size=0.1, random_state=SEED)

        splits = {"train": train_ids, "val": val_ids, "test": test_ids}

        for split_name, split_ids in splits.items():
            print(f"Processing split {split_name} with {len(split_ids)} segments")
            for segment_id in tqdm(split_ids, desc=f"Preprocessing {split_name}"):
                try:
                    # Kiểm tra sự tồn tại của segment_id trong tất cả các phương thức
                    if (segment_id not in dataset["language"].keys() or
                        segment_id not in dataset["acoustic"].keys() or
                        segment_id not in dataset["visual"].keys() or
                        segment_id not in dataset["labels"].keys()):
                        print(f"Skipping segment {segment_id}: missing features")
                        continue

                    text_features = dataset["language"][segment_id]["features"]
                    audio_features = dataset["acoustic"][segment_id]["features"]
                    visual_features = dataset["visual"][segment_id]["features"]
                    label = dataset["labels"][segment_id]["features"]

                    text_str = " ".join([word[0].decode('utf-8') if isinstance(word[0], bytes) else str(word[0]) for word in text_features])

                    text_embedding = self.extract_text_features(text_str)
                    audio_embedding = self.extract_audio_features(audio_features)
                    visual_embedding = self.extract_visual_features(visual_features)
                    sentiment_score = np.mean(label)

                    if np.isnan(sentiment_score):
                        continue
                    self.data[split_name]["language"].append(text_embedding)
                    self.data[split_name]["acoustic"].append(audio_embedding)
                    self.data[split_name]["visual"].append(visual_embedding)
                    self.data[split_name]["labels"].append(sentiment_score)
                except Exception as e:
                    print(f"Error while processing segment {segment_id}: {e}")
                    continue

            self.metadata[f"{split_name}_samples"] = len(self.data[split_name]["labels"])
            print(f"Split {split_name} has {self.metadata[f'{split_name}_samples']} samples")
        print(f"Keys in self.data[{split_name}]: {list(self.data[split_name].keys())}")
        print("Dataset processing complete")
        return True

    def save_processed_data(self):
        print("Saving processed data...")
        for split_name in ["train", "val", "test"]:
            for modality in ["language", "acoustic", "visual", "labels"]:
                self.data[split_name][modality] = np.array(self.data[split_name][modality])
                if np.any(np.isnan(self.data[split_name][modality])):
                    self.data[split_name][modality] = np.nan_to_num(self.data[split_name][modality], nan=0.0)

            split_file = self.processed_data_dir / f"{split_name}_data.pkl"
            with open(split_file, "wb") as f:
                pickle.dump(self.data[split_name], f)
            print(f"Saved {split_name} data to {split_file}")

        metadata_file = self.processed_data_dir / "metadata.pkl"
        with open(metadata_file, "wb") as f:
            pickle.dump(self.metadata, f)
        print(f"Saved metadata to {metadata_file}")

# CrossAttention (giữ nguyên từ mã gốc)
class CrossAttention(nn.Module):
    def __init__(self, query_dim, key_dim, value_dim, hidden_dim, num_heads=4, dropout_rate=0.1):
        super(CrossAttention, self).__init__()
        self.num_heads = num_heads
        self.hidden_dim = hidden_dim
        self.head_dim = hidden_dim // num_heads
        assert self.head_dim * num_heads == hidden_dim, "Hidden dimension must be divisible by number of heads"

        self.query_proj = nn.Linear(query_dim, hidden_dim)
        self.key_proj = nn.Linear(key_dim, hidden_dim)
        self.value_proj = nn.Linear(value_dim, hidden_dim)
        self.output_proj = nn.Linear(hidden_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, query, key, value):
        batch_size = query.size(0)
        query = self.query_proj(query).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        key = self.key_proj(key).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        value = self.value_proj(value).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)

        attention_scores = torch.matmul(query, key.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attention_weights = F.softmax(attention_scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        attended_values = torch.matmul(attention_weights, value)
        attended_values = attended_values.transpose(1, 2).contiguous().view(batch_size, -1, self.hidden_dim)
        output = self.output_proj(attended_values)

        return output

# MultimodalCrossAttention (giữ nguyên từ mã gốc)
class MultimodalCrossAttention(nn.Module):
    def __init__(self, text_dim, audio_dim, visual_dim, hidden_dim=128, num_heads=4, dropout_rate=0.1):
        super(MultimodalCrossAttention, self).__init__()
        self.text_proj = nn.Linear(text_dim, hidden_dim)
        self.audio_proj = nn.Linear(audio_dim, hidden_dim)
        self.visual_proj = nn.Linear(visual_dim, hidden_dim)

        self.audio_visual_attn = CrossAttention(hidden_dim, hidden_dim, hidden_dim, hidden_dim, num_heads, dropout_rate)
        self.visual_audio_attn = CrossAttention(hidden_dim, hidden_dim, hidden_dim, hidden_dim, num_heads, dropout_rate)
        self.audio_text_attn = CrossAttention(hidden_dim, hidden_dim, hidden_dim, hidden_dim, num_heads, dropout_rate)
        self.text_audio_attn = CrossAttention(hidden_dim, hidden_dim, hidden_dim, hidden_dim, num_heads, dropout_rate)
        self.visual_text_attn = CrossAttention(hidden_dim, hidden_dim, hidden_dim, hidden_dim, num_heads, dropout_rate)
        self.text_visual_attn = CrossAttention(hidden_dim, hidden_dim, hidden_dim, hidden_dim, num_heads, dropout_rate)

        self.fusion_layer = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )
        self.output_proj = nn.Linear(hidden_dim, 1)

    def forward(self, audio_features, visual_features, text_features):
        batch_size = audio_features.size(0)
        logger.debug(f"MultimodalCrossAttention input shapes: audio={audio_features.shape}, visual={visual_features.shape}, text={text_features.shape}")

        audio_proj = self.audio_proj(audio_features)
        visual_proj = self.visual_proj(visual_features)
        text_proj = self.text_proj(text_features)
        logger.debug(f"Shapes after projection: audio={audio_proj.shape}, visual={visual_proj.shape}, text={text_proj.shape}")

        if len(audio_proj.shape) == 2:
            audio_proj = audio_proj.unsqueeze(1)
            visual_proj = visual_proj.unsqueeze(1)
            text_proj = text_proj.unsqueeze(1)
        logger.debug(f"Shapes after unsqueeze: audio={audio_proj.shape}, visual={visual_proj.shape}, text={text_proj.shape}")

        audio_visual_attn = self.audio_visual_attn(audio_proj, visual_proj, visual_proj)
        visual_audio_attn = self.visual_audio_attn(visual_proj, audio_proj, audio_proj)
        audio_text_attn = self.audio_text_attn(audio_proj, text_proj, text_proj)
        text_audio_attn = self.text_audio_attn(text_proj, audio_proj, audio_proj)
        visual_text_attn = self.visual_text_attn(visual_proj, text_proj, text_proj)
        text_visual_attn = self.text_visual_attn(text_proj, visual_proj, visual_proj)

        audio_context = audio_proj + audio_visual_attn + audio_text_attn
        visual_context = visual_proj + visual_audio_attn + visual_text_attn
        text_context = text_proj + text_audio_attn + text_visual_attn

        audio_context = audio_context.squeeze(1) if audio_context.size(1) == 1 else audio_context.mean(dim=1)
        visual_context = visual_context.squeeze(1) if visual_context.size(1) == 1 else visual_context.mean(dim=1)
        text_context = text_context.squeeze(1) if text_context.size(1) == 1 else text_context.mean(dim=1)

        concat_features = torch.cat([audio_context, visual_context, text_context], dim=1)

        if concat_features.size(1) != 384:
            raise ValueError(f"Feature size mismatch: {concat_features.shape}")

        fused_features = self.fusion_layer(concat_features)
        sentiment = self.output_proj(fused_features)

        if torch.isnan(sentiment).any():
            sentiment = torch.nan_to_num(sentiment, nan=0.0)

        return fused_features, sentiment

# LSTM Audio Encoder
class LSTMAudioEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_layers=2, dropout_rate=0.3):
        super(LSTMAudioEncoder, self).__init__()
        self.input_projection = nn.Linear(input_dim, hidden_dim)
        self.lstm = nn.LSTM(hidden_dim, hidden_dim, num_layers=num_layers, batch_first=True, dropout=dropout_rate if num_layers > 1 else 0)
        self.output_projection = nn.Linear(hidden_dim, 1)
        self.dropout = nn.Dropout(dropout_rate)

    def get_encoded_features(self, x):
        x = self.input_projection(x)
        if len(x.shape) == 2:
            x = x.unsqueeze(1)  # Thêm chiều time-step
        x, (hn, cn) = self.lstm(x)
        encoded = hn[-1]  # Lấy trạng thái ẩn cuối cùng
        return encoded

    def forward(self, x):
        encoded = self.get_encoded_features(x)
        sentiment = self.output_projection(encoded)
        return encoded, sentiment

# LSTM Visual Encoder
class LSTMVisualEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_layers=2, dropout_rate=0.3):
        super(LSTMVisualEncoder, self).__init__()
        self.input_projection = nn.Linear(input_dim, hidden_dim)
        self.lstm = nn.LSTM(hidden_dim, hidden_dim, num_layers=num_layers, batch_first=True, dropout=dropout_rate if num_layers > 1 else 0)
        self.output_projection = nn.Linear(hidden_dim, 1)
        self.dropout = nn.Dropout(dropout_rate)

    def get_encoded_features(self, x):
        x = self.input_projection(x)
        if len(x.shape) == 2:
            x = x.unsqueeze(1)  # Thêm chiều time-step
        x, (hn, cn) = self.lstm(x)
        encoded = hn[-1]  # Lấy trạng thái ẩn cuối cùng
        return encoded

    def forward(self, x):
        encoded = self.get_encoded_features(x)
        sentiment = self.output_projection(encoded)
        return encoded, sentiment

# LSTM Text Encoder
class LSTMTextEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_layers=2, dropout_rate=0.3):
        super(LSTMTextEncoder, self).__init__()
        self.input_projection = nn.Linear(input_dim, hidden_dim)
        self.lstm = nn.LSTM(hidden_dim, hidden_dim, num_layers=num_layers, batch_first=True, dropout=dropout_rate if num_layers > 1 else 0)
        self.output_projection = nn.Linear(hidden_dim, 1)
        self.dropout = nn.Dropout(dropout_rate)

    def get_encoded_features(self, x):
        logger.debug(f"LSTMTextEncoder input shape: {x.shape}")
        x = self.input_projection(x)
        if len(x.shape) == 2:
            x = x.unsqueeze(1)  # Thêm chiều time-step
        logger.debug(f"LSTMTextEncoder after projection: {x.shape}")
        x, (hn, cn) = self.lstm(x)
        encoded = hn[-1]  # Lấy trạng thái ẩn cuối cùng
        logger.debug(f"LSTMTextEncoder output shape: {encoded.shape}")
        return encoded

    def forward(self, x):
        encoded = self.get_encoded_features(x)
        sentiment = self.output_projection(encoded)
        return encoded, sentiment

# LSTM Fusion Model
class LSTMFusionModel(nn.Module):
    def __init__(self, audio_dim, visual_dim, text_dim, hidden_dim=128, num_heads=4, num_layers=2, dropout_rate=0.3):
        super(LSTMFusionModel, self).__init__()
        self.audio_encoder = LSTMAudioEncoder(audio_dim, hidden_dim, num_layers, dropout_rate)
        self.visual_encoder = LSTMVisualEncoder(visual_dim, hidden_dim, num_layers, dropout_rate)
        self.text_encoder = LSTMTextEncoder(text_dim, hidden_dim, num_layers, dropout_rate)
        self.fusion_module = MultimodalCrossAttention(hidden_dim, hidden_dim, hidden_dim, hidden_dim, num_heads, dropout_rate)
        self.output_proj = nn.Linear(hidden_dim, 1)

    def forward(self, features):
        audio_features = features["acoustic"]
        visual_features = features["visual"]
        text_features = features["language"]
        logger.debug(f"LSTMFusionModel input shapes: audio={audio_features.shape}, visual={visual_features.shape}, text={text_features.shape}")
        audio_encoded = self.audio_encoder.get_encoded_features(audio_features)
        visual_encoded = self.visual_encoder.get_encoded_features(visual_features)
        text_encoded = self.text_encoder.get_encoded_features(text_features)
        logger.debug(f"LSTMFusionModel encoded shapes: audio={audio_encoded.shape}, visual={visual_encoded.shape}, text={text_encoded.shape}")
        fused_features, sentiment = self.fusion_module(audio_encoded, visual_encoded, text_encoded)
        logger.debug(f"LSTMFusionModel output shape: {sentiment.shape}")
        return sentiment

# Audio Sentiment Model (giữ nguyên)
class AudioSentimentModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, dropout_rate=0.3):
        super(AudioSentimentModel, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.batch_norm1 = nn.BatchNorm1d(hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.batch_norm2 = nn.BatchNorm1d(hidden_dim // 2)
        self.fc3 = nn.Linear(hidden_dim // 2, 1)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        x = self.fc1(x)
        x = self.batch_norm1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = self.batch_norm2(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc3(x)
        return x

# Text Sentiment Model (giữ nguyên)
class TextSentimentModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, dropout_rate=0.3):
        super(TextSentimentModel, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.fc3 = nn.Linear(hidden_dim // 2, 1)
        self.dropout = nn.Dropout(dropout_rate)
        self.batch_norm1 = nn.BatchNorm1d(hidden_dim)
        self.batch_norm2 = nn.BatchNorm1d(hidden_dim // 2)

    def forward(self, x):
        x = self.fc1(x)
        x = self.batch_norm1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = self.batch_norm2(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc3(x)
        return x

# Visual Sentiment Model (giữ nguyên)
class VisualSentimentModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, dropout_rate=0.3):
        super(VisualSentimentModel, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.fc3 = nn.Linear(hidden_dim // 2, 1)
        self.dropout = nn.Dropout(dropout_rate)
        self.batch_norm1 = nn.BatchNorm1d(hidden_dim)
        self.batch_norm2 = nn.BatchNorm1d(hidden_dim // 2)

    def forward(self, x):
        x = self.fc1(x)
        x = self.batch_norm1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = self.batch_norm2(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc3(x)
        return x

# Late Fusion Model (giữ nguyên)
class LateFusionModel(nn.Module):
    def __init__(self, audio_model, visual_model, text_model, fusion_weights=None):
        super(LateFusionModel, self).__init__()
        self.audio_model = audio_model
        self.visual_model = visual_model
        self.text_model = text_model
        for model in [self.audio_model, self.visual_model, self.text_model]:
            for param in model.parameters():
                param.requires_grad = False
        if fusion_weights is None:
            self.fusion_weights = nn.Parameter(torch.ones(2) / 2)
        else:
            self.fusion_weights = torch.tensor(fusion_weights)

    def forward(self, features):
        audio_features = features["acoustic"]
        visual_features = features["visual"]
        text_features = features["language"]
        audio_pred = self.audio_model(audio_features)
        visual_pred = self.visual_model(visual_features)
        text_pred = self.text_model(text_features)
        weights = F.softmax(self.fusion_weights, dim=0)
        combined_pred = weights[0] * audio_pred + weights[1] * visual_pred
        return combined_pred

# Trainer (giữ nguyên)
class Trainer:
    def __init__(self, model, train_loader, val_loader, test_loader=None, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY):
        self.model = model.to(DEVICE)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.test_loader = test_loader
        self.optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        self.criterion = nn.MSELoss()
        self.model_dir = Path("/content/models")
        self.model_dir.mkdir(exist_ok=True, parents=True)
        self.experiment_name = f"fusion_av_{time.strftime('%Y%m%d_%H%M%S')}"
        self.best_val_loss = float("inf")
        self.best_epoch = 0
        self.patience_counter = 0

    def train_epoch(self, epoch):
        self.model.train()
        epoch_loss = 0.0
        pbar = tqdm(total=len(self.train_loader), desc=f"Epoch {epoch}")

        for batch in self.train_loader:
            inputs = {k: v.to(DEVICE) for k, v in batch.items() if k != "label"}
            labels = batch["label"].to(DEVICE)
            self.optimizer.zero_grad()
            outputs = self.model(inputs)
            outputs = torch.nan_to_num(outputs, nan=0.0)
            loss = self.criterion(outputs.squeeze(), labels)
            if torch.isnan(loss):
                continue
            loss.backward()
            self.optimizer.step()
            epoch_loss += loss.item()
            pbar.update(1)
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        pbar.close()
        return epoch_loss / len(self.train_loader)

    def validate(self, epoch):
        self.model.eval()
        val_loss = 0.0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for batch in self.val_loader:
                inputs = {k: v.to(DEVICE) for k, v in batch.items() if k != "label"}
                labels = batch["label"].to(DEVICE)
                outputs = self.model(inputs)
                outputs = torch.nan_to_num(outputs, nan=0.0)
                loss = self.criterion(outputs.squeeze(), labels)
                val_loss += loss.item()
                all_preds.append(outputs.squeeze().cpu().numpy())
                all_labels.append(labels.cpu().numpy())

        avg_loss = val_loss / len(self.val_loader)
        all_preds = np.concatenate(all_preds)
        all_labels = np.concatenate(all_labels)

        if np.any(np.isnan(all_preds)) or np.any(np.isnan(all_labels)):
            all_preds = np.nan_to_num(all_preds, nan=0.0)
            all_labels = np.nan_to_num(all_labels, nan=0.0)

        metrics = {
            "mae": mean_absolute_error(all_labels, all_preds),
            "corr": np.corrcoef(all_preds, all_labels)[0, 1] if np.std(all_preds) > 0 and np.std(all_labels) > 0 else 0.0,
            "binary_acc": accuracy_score((all_labels > 0).astype(int), (all_preds > 0).astype(int)),
            "binary_f1": f1_score((all_labels > 0).astype(int), (all_preds > 0).astype(int))
        }
        return avg_loss, metrics

    def save_checkpoint(self, epoch, val_loss, is_best=False):
        checkpoint = {
            "epoch": epoch,
            "model_state_dict": self.model.state_dict(),
            "optimizer_state_dict": self.optimizer.state_dict(),
            "val_loss": val_loss,
        }
        checkpoint_path = self.model_dir / f"{self.experiment_name}_epoch_{epoch}.pt"
        torch.save(checkpoint, checkpoint_path)
        print(f"Saved checkpoint to {checkpoint_path}")
        if is_best:
            best_path = self.model_dir / f"{self.experiment_name}_best.pt"
            torch.save(checkpoint, best_path)
            print(f"Best model saved to {best_path}")

    def train(self, num_epochs=NUM_EPOCHS, patience=3, eval_every=1):
        train_losses = []
        val_losses = []
        val_metrics_list = []

        for epoch in range(1, num_epochs + 1):
            train_loss = self.train_epoch(epoch)
            train_losses.append(train_loss)
            print(f"Epoch {epoch} - Training Loss: {train_loss:.4f}")

            if epoch % eval_every == 0:
                val_loss, val_metrics = self.validate(epoch)
                val_losses.append(val_loss)
                val_metrics_list.append(val_metrics)
                print(f"Epoch {epoch} - Validate Loss: {val_loss:.4f}, MAE: {val_metrics['mae']:.4f}, Corr: {val_metrics['corr']:.4f}, Binary Acc: {val_metrics['binary_acc']:.4f}, Binary F1: {val_metrics['binary_f1']:.4f}")

                if val_loss < self.best_val_loss:
                    self.best_val_loss = val_loss
                    self.best_epoch = epoch
                    self.save_checkpoint(epoch, val_loss, is_best=True)
                    self.patience_counter = 0
                else:
                    self.patience_counter += 1
                    if self.patience_counter >= patience:
                        print(f"Early stopping at epoch {epoch}")
                        break

                if epoch % 2 == 0 or epoch == num_epochs:
                    self.save_checkpoint(epoch, val_loss)

        return train_losses, val_losses, val_metrics_list

# Hàm trực quan hóa (giữ nguyên)
def plot_training_curves(train_losses, val_losses, metrics, save_path, model_name):
    plt.figure(figsize=(10, 8))
    plt.subplot(2, 1, 1)
    plt.plot(train_losses, label="Training Loss", color="blue")
    plt.plot(val_losses, label="Validate Loss", color="red")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"Training and Validation Loss - {model_name}")
    plt.legend()
    plt.grid(True)

    plt.subplot(2, 1, 2)
    for metric_name in ["mae", "corr", "binary_acc", "binary_f1"]:
        plt.plot([m[metric_name] for m in metrics], label=metric_name.capitalize())
    plt.xlabel("Epoch")
    plt.ylabel("Metric Value")
    plt.title(f"Validation Metrics - {model_name}")
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    print(f"Saved training curve for {model_name} at {save_path}")
    plt.close()

def plot_results_summary(results, save_path):
    metrics = ["mae", "corr", "binary_acc", "binary_f1"]
    model_names = list(results.keys())
    metric_values = {metric: [results[model][metric] for model in model_names] for metric in metrics}

    plt.figure(figsize=(12, 6))
    x = np.arange(len(model_names))
    width = 0.2

    for i, metric in enumerate(metrics):
        plt.bar(x + i * width, metric_values[metric], width, label=metric.capitalize())

    plt.xlabel("Model")
    plt.ylabel("Metric Value")
    plt.title("Model Comparison")
    plt.xticks(x + width * 1.5, model_names)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    print(f"Model comparison chart saved to {save_path}")
    plt.close()

# Tạo và huấn luyện mô hình
preprocessor = MOSEIPreprocessor(RAW_DATA_DIR, PROCESSED_DATA_DIR)
print("Preprocessing dataset...")
if not preprocessor.process_dataset():
    print("Cannot process dataset")
    exit()

print("Saving processed data...")
preprocessor.save_processed_data()

train_dataset = MOSEIDataset(preprocessor.data["train"])
val_dataset = MOSEIDataset(preprocessor.data["val"])
test_dataset = MOSEIDataset(preprocessor.data["test"])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=True)
print(f"Created DataLoaders: Train={len(train_loader.dataset)}, Val={len(val_loader.dataset)}, Test={len(test_loader.dataset)}")

audio_model = AudioSentimentModel(input_dim=AUDIO_FEATURE_SIZE).to(DEVICE)
visual_model = VisualSentimentModel(input_dim=VISUAL_FEATURE_SIZE).to(DEVICE)
text_model = TextSentimentModel(input_dim=TEXT_EMBEDDING_DIM).to(DEVICE)

models = {
    "LateFusion": LateFusionModel(audio_model, visual_model, text_model).to(DEVICE),
    "LSTMFusion": LSTMFusionModel(
        audio_dim=AUDIO_FEATURE_SIZE,
        visual_dim=VISUAL_FEATURE_SIZE,
        text_dim=TEXT_EMBEDDING_DIM
    ).to(DEVICE)
}

results = {}
plot_dir = Path("/content/plots")
plot_dir.mkdir(exist_ok=True, parents=True)

for model_name, model in models.items():
    print(f"Training model {model_name}...")
    trainer = Trainer(model, train_loader, val_loader, test_loader)
    train_losses, val_losses, val_metrics_list = trainer.train(num_epochs=NUM_EPOCHS, patience=3)

    best_metrics = val_metrics_list[-1]
    results[model_name] = best_metrics

    plot_training_curves(
        train_losses, val_losses, val_metrics_list,
        save_path=str(plot_dir / f"training_curves_{model_name.lower()}_av.png"),
        model_name=model_name
    )

print("Creating model comparison chart...")
plot_results_summary(results, str(plot_dir / "model_comparison_av.png"))

best_model_name = min(results, key=lambda x: results[x]["mae"])
print(f"Evaluating the best model ({best_model_name}) on test set...")
trainer = Trainer(models[best_model_name], train_loader, val_loader, test_loader)
test_loss, test_metrics = trainer.validate(0)
print(f"Test results for {best_model_name}: Loss: {test_loss:.4f}, MAE: {test_metrics['mae']:.4f}, Corr: {test_metrics['corr']:.4f}, Binary Acc: {test_metrics['binary_acc']:.4f}, Binary F1: {test_metrics['binary_f1']:.4f}")

100%|██████████| 29.1G/29.1G [16:21<00:00, 31.8MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/samarwarsi/cmu-mosei/versions/1


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Using device: cuda
Preprocessing dataset...
Preprocessing dataset...
Loading aligned data...
<Success>: Computational sequence read from file /root/.cache/kagglehub/datasets/samarwarsi/cmu-mosei/versions/1/CMU-MOSEI/languages/CMU_MOSEI_TimestampedWords.csd ...
<Status>: Checking the integrity of the data in <b'"words"'> computational sequence ...


...

Epoch 2 - Training Loss: 0.0532
Epoch 2 - Validate Loss: 0.0539, MAE: 0.1873, Corr: 0.0000, Binary Acc: 0.0352, Binary F1: 0.0000
Saved checkpoint to /content/models/fusion_av_20250628_082202_epoch_2.pt


Epoch 3: 100%|██████████| 74/74 [00:01<00:00, 41.40it/s, loss=0.0437]


Epoch 3 - Training Loss: 0.0532
Epoch 3 - Validate Loss: 0.0539, MAE: 0.1873, Corr: 0.0000, Binary Acc: 0.0352, Binary F1: 0.0000


Epoch 4: 100%|██████████| 74/74 [00:01<00:00, 42.91it/s, loss=0.0725]


Epoch 4 - Training Loss: 0.0533
Epoch 4 - Validate Loss: 0.0539, MAE: 0.1873, Corr: 0.0000, Binary Acc: 0.0352, Binary F1: 0.0000
Early stopping at epoch 4
Saved training curve for LSTMFusion at /content/plots/training_curves_lstmfusion_av.png
Creating model comparison chart...
Model comparison chart saved to /content/plots/model_comparison_av.png
Evaluating the best model (LateFusion) on test set...
Test results for LateFusion: Loss: 0.0539, MAE: 0.1873, Corr: 0.0000, Binary Acc: 0.0352, Binary F1: 0.0000
